# DATA WRANGLING

## SETUP AND IMPORT

In [1]:
import duckdb
import pandas as pd
import sqlite3
from pathlib import Path
import holidays
import os

os.chdir("/Users/jakoberhard/Library/CloudStorage/GoogleDrive-jakanterh@gmail.com/My Drive/uni/python/TBA_project")

con = duckdb.connect("data/train.duckdb")

con.execute("""
CREATE OR REPLACE TABLE train_delay AS
            SELECT * FROM
            read_parquet('data/train_delay_with_weather.parquet')
            """)

df_raw = con.execute("SELECT * FROM train_delay").fetchdf()

## IMPORT DATA

## DATA INSPECTION

In [2]:
print(df_raw.info())                      
# print(df_raw.head(5))                      
# print(df_raw["delay_in_min"].describe())  
# print(df_raw["train_type"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2175996 entries, 0 to 2175995
Data columns (total 15 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ride_id            int64         
 1   train_name         object        
 2   train_type         object        
 3   station_current    object        
 4   station_dest       object        
 5   canceled           bool          
 6   arrival_planned    datetime64[us]
 7   arrival_real       datetime64[us]
 8   departure_planned  datetime64[us]
 9   departure_real     datetime64[us]
 10  delay              int32         
 11  temperature        float64       
 12  precipitation      float64       
 13  wind_speed         float64       
 14  weather_status     object        
dtypes: bool(1), datetime64[us](4), float64(3), int32(1), int64(1), object(5)
memory usage: 226.2+ MB
None


## DATA CLEANING

In [3]:
### FILTER TRAIN TYPES (ONLY ICE AND IC)
def filter_train_type(df):
    output_df = df[df["train_type"].str.contains("ICE|IC", case=False, na=False)].copy()
    return output_df

df_cleaned = filter_train_type(df_raw)

### REMOVE RIDES THAT WERE CANCELED
df_cleaned = (
    df_cleaned.groupby("ride_id")
      .filter(lambda g: not g["canceled"].any())
)

### REMOVE UNECESSARY COLUMNS
df_cleaned = df_cleaned.drop(columns=["weather_status", "canceled"])

### RENAME COLUMNS
df_cleaned = df_cleaned.rename(columns={
    "station_name": "station_current",
    "final_destination_station": "station_dest"})

# REARRANGE COLUMNS
df_cleaned = df_cleaned.loc[:, ["ride_id", "train_name", "train_type", "station_current", "station_dest",
               "departure_planned", "departure_real", "arrival_planned", "arrival_real",
                "temperature", "precipitation", "wind_speed", "delay"]]


print(df_cleaned.columns.tolist())
print(df_cleaned.info())    


['ride_id', 'train_name', 'train_type', 'station_current', 'station_dest', 'departure_planned', 'departure_real', 'arrival_planned', 'arrival_real', 'temperature', 'precipitation', 'wind_speed', 'delay']
<class 'pandas.core.frame.DataFrame'>
Index: 1842266 entries, 0 to 2175995
Data columns (total 13 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ride_id            int64         
 1   train_name         object        
 2   train_type         object        
 3   station_current    object        
 4   station_dest       object        
 5   departure_planned  datetime64[us]
 6   departure_real     datetime64[us]
 7   arrival_planned    datetime64[us]
 8   arrival_real       datetime64[us]
 9   temperature        float64       
 10  precipitation      float64       
 11  wind_speed         float64       
 12  delay              int32         
dtypes: datetime64[us](4), float64(3), int32(1), int64(1), object(4)
memory usage: 189.7+ MB
None


## NA ANALYSIS

In [4]:
# number of na per column
print(df_cleaned.isnull().sum(axis = 0))

# missings in time_current_departure_planned and time_current_arrival_plannned
# are due to first or last station that do not have either arrival time (start station) or departure time (destination station)
len(df_cleaned["ride_id"].unique())

ride_id                   0
train_name                0
train_type                0
station_current           0
station_dest              0
departure_planned    200154
departure_real       200031
arrival_planned      200154
arrival_real         200036
temperature               0
precipitation             0
wind_speed                0
delay                     0
dtype: int64


200154

## DATA-TYPES --> DO IN SCI-KIT

In [ ]:
# categorical variables
categorical_cols = [
    "journey_id",
    "train_name",
    "train_type",
    "station_current",
    "station_dest"]
df[categorical_cols] = df[categorical_cols].astype("category")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3551715 entries, 0 to 3551714
Data columns (total 11 columns):
 #   Column                          Dtype         
---  ------                          -----         
 0   id                              int64         
 1   journey_id                      category      
 2   train_name                      category      
 3   train_type                      category      
 4   station_current                 category      
 5   station_dest                    category      
 6   time_current_departure_planned  datetime64[ns]
 7   time_current_departure_real     datetime64[ns]
 8   time_current_arrival_planned    datetime64[ns]
 9   delay                           int32         
 10  canceled                        bool          
dtypes: bool(1), category(5), datetime64[ns](3), int32(1), int64(1)
memory usage: 185.0 MB


## DEFINE STATION_START

In [5]:
### REASON: ALSO EXISTS IN API DATA
### AFTER THAT WE CAN APPLY FUNCTIONS TO BOTH DATASETS

# grouping
grouped = df_cleaned.groupby("ride_id")

# create station_start
df_cleaned["station_start"] = grouped["station_current"].transform("first")

## RIDE-RELATED

### STATION NAMES, TIME AND STOPS

In [ ]:
import numpy as np
import holidays



def create_features(df):

    # create copy: do not overwrite input df
    df = df.copy()

    # check that all date variables have the correct format
    datetime_cols = ["departure_planned", "arrival_planned", "departure_real", "arrival_real"]
    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col])


    ### RIDE RELATED 

    # prepare grouping
    grouped = df.groupby("ride_id")

    # number of stops
    df["stops_total"] = grouped["station_current"].transform("count")

    # time departure and arrival
    df["departure_planned_start"] = grouped["departure_planned"].transform("first")
    df["arrival_planned_dest"] = grouped["arrival_planned"].transform("last")

    # travel time
    df["travel_time"] = (df["arrival_planned_dest"] - df["departure_planned_start"]).dt.total_seconds() / 60

    # weekday and month
    df["weekday"] = df["departure_planned_start"].dt.weekday
    df["month"] = df["departure_planned_start"].dt.month

    # feast
    de_holidays = holidays.Germany()
    df["feast"] = df["departure_planned_start"].dt.date.apply(lambda x: x in de_holidays)


    ### STATION RELATED

    # hour
    df["hour"] = np.where(
        df["departure_real"].notna(), 
        df["departure_real"].dt.hour, 
        df["arrival_real"].dt.hour)
    
    # dwell-time planned
    df["dwell_time_planned"] = (df["departure_planned"] - df["arrival_planned"]).dt.total_seconds() / 60


    ### SEQUENCE RELATED

    

    return df


df_features = create_features(df_cleaned)

### HISTORICAL AGGREGATION

In [15]:
# function to add expanding features
def add_expanding_features(df, group_cols, prefix):
    df = df.sort_values(group_cols + ["time_current_departure_real"]) # sort by real time because otherwise expanding stats might include info from future

    grouped = df.groupby(group_cols, sort=False)["delay"]

    df[f"{prefix}_avg"] = grouped.transform(lambda x: x.shift().expanding().mean())
    df[f"{prefix}_std"] = grouped.transform(lambda x: x.shift().expanding().std())
    df[f"{prefix}_q90"] = grouped.transform(lambda x: x.shift().expanding().quantile(0.9))
    df[f"{prefix}_count"] = grouped.transform(lambda x: x.shift().expanding().count()
)

    return df

# derive time features from real time
features_station["hour"] = features_station["time_current_departure_real"].dt.hour
features_station["weekday"] = features_station["time_current_departure_real"].dt.dayofweek

# apply expanding features
features_station = add_expanding_features(features_station, ["station_current"], "hist_delay_station")   
features_station = add_expanding_features(features_station, ["station_current", "hour"], "hist_delay_station_hour")  
features_station = add_expanding_features(features_station, ["station_current", "weekday"], "hist_delay_station_day")  

/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/1970335331.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby(group_cols, sort=False)["delay"]
/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/1970335331.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby(group_cols, sort=False)["delay"]
/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/1970335331.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or ob

### DROP UNNECESSARY VARIABLES

In [16]:
features_station.columns.tolist()

['id',
 'journey_id',
 'train_name',
 'train_type',
 'station_current',
 'station_dest',
 'time_current_departure_planned',
 'time_current_departure_real',
 'time_current_arrival_planned',
 'delay',
 'canceled',
 'dwell_time_planned',
 'hour',
 'weekday',
 'hist_delay_station_avg',
 'hist_delay_station_std',
 'hist_delay_station_q90',
 'hist_delay_station_count',
 'hist_delay_station_hour_avg',
 'hist_delay_station_hour_std',
 'hist_delay_station_hour_q90',
 'hist_delay_station_hour_count',
 'hist_delay_station_day_avg',
 'hist_delay_station_day_std',
 'hist_delay_station_day_q90',
 'hist_delay_station_day_count']

In [18]:
columns_to_drop = [
    "journey_id",
    "train_name",
    "train_type",
    "station_current",
    "station_dest",
    "time_current_arrival_planned",
    "time_current_departure_planned",
    "time_current_departure_real",
    "delay",
    "canceled",
    "weekday"]
features_station.drop(columns=columns_to_drop, inplace=True)

## SEQUENCE-RELATED

### JOIN INFORMATION

In [23]:
# join features back to main dataframe
df_full = df.merge(
    features_ride,
    on = "journey_id",
    how = "left"
)
df_full = df_full.merge(
    features_station,
    on = "id",
    how = "left"
)
print(df_full.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3551715 entries, 0 to 3551714
Data columns (total 35 columns):
 #   Column                          Dtype         
---  ------                          -----         
 0   id                              int64         
 1   journey_id                      category      
 2   train_name                      category      
 3   train_type                      category      
 4   station_current                 category      
 5   station_dest                    category      
 6   time_current_departure_planned  datetime64[ns]
 7   time_current_departure_real     datetime64[ns]
 8   time_current_arrival_planned    datetime64[ns]
 9   delay                           int32         
 10  canceled                        bool          
 11  station_start                   category      
 12  stops_total                     int64         
 13  time_start_planned              datetime64[ns]
 14  time_dest_planned               datetime64[ns]
 15

### SEQUENCE VARS

In [24]:
# station role: start, mid, end
df_full["station_role"] = df_full.apply(
    lambda row: "start" if row["station_current"] == row["station_start"] else (
        "end" if row["station_current"] == row["station_dest"] else "mid"), axis=1).astype("category")    

# stops index in journey
df_full["stop_index"] = df_full.groupby("journey_id").cumcount() + 1    

# stops remaining to destination
df_full["stops_remaining"] = df_full["stops_total"] - df_full["stop_index"]

# time since start planned
df_full["time_since_start_planned"] = (
    df_full["time_current_arrival_planned"]
    - df_full["time_start_planned"]
).dt.total_seconds() / 60   

# time to destination planned
df_full["time_to_dest_planned"] = (
    df_full["time_dest_planned"]
    - df_full["time_current_departure_planned"]
).dt.total_seconds() / 60

# share ride time
df_full["share_ride_time_completed"] = (
    df_full["time_since_start_planned"] / df_full["travel_time"]
)

# share ride stations
df_full["share_ride_stations"] = (
    df_full["stop_index"] / df_full["stops_total"]
)

/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/2729890663.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_full["stop_index"] = df_full.groupby("journey_id").cumcount() + 1


## SAVE DATASET

In [ ]:
# save final dataframe
OUTPUT_DIR = BASE_DIR / "data" / "train_delay_processed.duckdb"
with duckdb.connect(OUTPUT_DIR) as con:
    con.execute("DROP TABLE IF EXISTS train_delay_processed")
    con.execute("CREATE TABLE train_delay_processed AS SELECT * FROM df_final") 